## 1. Inisialisasi Environment dan SparkSession

Inisialisasi `SparkSession` lokal sebagai entry point untuk mengeksekusi operasi DataFrame dan SQL di PySpark.

In [1]:
!pip install pyspark -q

import os
from datetime import datetime, timedelta
import numpy as np
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("LogisticsBottleneckAnalysis") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
spark

Spark Session Berhasil Dibuat: 4.0.4


## 2. Simulasi Dataset Logistik (`scan_events`)

Simulasi dataset logistik mencakup 10.000 paket unik yang didistribusikan ke 6 hub operasional: `HUB_JKT`, `HUB_SUB`, `HUB_BDG`, `HUB_MKS`, `HUB_MES`, dan `HUB_SRG`.

Untuk merefleksikan dinamika operasional nyata, dataset dirancang dengan komposisi skenario sebagai berikut:
* **85% Data Valid**: Pasangan event `ARRIVAL` dan `DEPARTURE` lengkap. Hub `HUB_MKS` dan `HUB_MES` disimulasikan memiliki rata-rata waktu tunggu lebih lama guna menguji deteksi bottleneck.
* **5% Kasus Missing DEPARTURE**: Paket tercatat masuk hub tetapi tidak memiliki catatan keluar.
* **4% Kasus Missing ARRIVAL**: Paket memiliki catatan keluar tanpa catatan kedatangan.
* **3% Kasus Duplicate Scan**: Pemindaian ganda (*double tap*) pada waktu yang identik.
* **3% Kasus Anomali Timestamp**: Waktu `DEPARTURE` tercatat lebih awal dari waktu `ARRIVAL`.

Identitas hub dikonversi eksplisit ke tipe data string Python standar (`str`) untuk memastikan inferensi skema DataFrame PySpark berjalan optimal.

In [2]:
np.random.seed(42)
num_packages = 10000

hubs = ["HUB_JKT", "HUB_SUB", "HUB_BDG", "HUB_MKS", "HUB_MES", "HUB_SRG"]
data = []
base_time = datetime(2025, 10, 7, 8, 0, 0)

for i in range(1, num_packages + 1):
    package_id = f"PKG{i:05d}"
    hub_id = str(np.random.choice(hubs))
    rand_val = np.random.rand()

    if rand_val < 0.85:
        arr_time = base_time + timedelta(minutes=float(np.random.randint(10, 5000)))
        if hub_id == "HUB_MKS":
            dwell_hours = np.random.exponential(scale=18.0)
        elif hub_id == "HUB_MES":
            dwell_hours = np.random.exponential(scale=12.0)
        else:
            dwell_hours = np.random.exponential(scale=4.0)

        dep_time = arr_time + timedelta(hours=dwell_hours)
        data.append((hub_id, package_id, "ARRIVAL", arr_time.strftime('%Y-%m-%d %H:%M:%S')))
        data.append((hub_id, package_id, "DEPARTURE", dep_time.strftime('%Y-%m-%d %H:%M:%S')))

    elif rand_val < 0.90:
        arr_time = base_time + timedelta(minutes=float(np.random.randint(10, 5000)))
        data.append((hub_id, package_id, "ARRIVAL", arr_time.strftime('%Y-%m-%d %H:%M:%S')))

    elif rand_val < 0.94:
        dep_time = base_time + timedelta(minutes=float(np.random.randint(100, 5000)))
        data.append((hub_id, package_id, "DEPARTURE", dep_time.strftime('%Y-%m-%d %H:%M:%S')))

    elif rand_val < 0.97:
        arr_time = base_time + timedelta(minutes=float(np.random.randint(10, 5000)))
        dep_time = arr_time + timedelta(hours=np.random.exponential(scale=4.0))
        data.append((hub_id, package_id, "ARRIVAL", arr_time.strftime('%Y-%m-%d %H:%M:%S')))
        data.append((hub_id, package_id, "ARRIVAL", arr_time.strftime('%Y-%m-%d %H:%M:%S')))
        data.append((hub_id, package_id, "DEPARTURE", dep_time.strftime('%Y-%m-%d %H:%M:%S')))

    else:
        arr_time = base_time + timedelta(minutes=float(np.random.randint(500, 5000)))
        dep_time = arr_time - timedelta(hours=float(np.random.randint(1, 5)))
        data.append((hub_id, package_id, "ARRIVAL", arr_time.strftime('%Y-%m-%d %H:%M:%S')))
        data.append((hub_id, package_id, "DEPARTURE", dep_time.strftime('%Y-%m-%d %H:%M:%S')))

columns = ["hub_id", "package_id", "event_type", "timestamp"]
sdf = spark.createDataFrame(data, schema=columns)
sdf = sdf.withColumn("timestamp", F.to_timestamp("timestamp"))

sdf.show(10, truncate=False)
print("Total Scan Events Generated:", sdf.count())

+-------+----------+----------+-------------------+
|hub_id |package_id|event_type|timestamp          |
+-------+----------+----------+-------------------+
|HUB_MKS|PKG00001  |ARRIVAL   |2025-10-09 23:02:00|
|HUB_MKS|PKG00001  |ARRIVAL   |2025-10-09 23:02:00|
|HUB_MKS|PKG00001  |DEPARTURE |2025-10-10 02:40:01|
|HUB_SUB|PKG00002  |ARRIVAL   |2025-10-10 09:56:00|
|HUB_SUB|PKG00002  |DEPARTURE |2025-10-10 12:23:33|
|HUB_MES|PKG00003  |ARRIVAL   |2025-10-07 10:20:00|
|HUB_MES|PKG00003  |DEPARTURE |2025-10-07 10:34:58|
|HUB_SUB|PKG00004  |ARRIVAL   |2025-10-09 00:43:00|
|HUB_SUB|PKG00004  |DEPARTURE |2025-10-09 01:31:09|
|HUB_MES|PKG00005  |ARRIVAL   |2025-10-09 16:35:00|
+-------+----------+----------+-------------------+
only showing top 10 rows
Total Scan Events Generated: 19482


## 3. Pemeriksaan Kualitas Data dan Pembersihan (Data Quality Check)

Pemeriksaan kualitas data dan penataan ulang struktur data:
1. **Deduplikasi Scan**: Mengidentifikasi dan menghapus event duplikat dengan `dropDuplicates` pada kombinasi `hub_id`, `package_id`, `event_type`, dan `timestamp`.
2. **Pivot Event**: Mengubah baris `ARRIVAL` dan `DEPARTURE` menjadi kolom terpisah per paket dan hub untuk mempermudah perbandingan waktu.

In [3]:
total_raw_events = sdf.count()
duplicates = sdf.groupBy("hub_id", "package_id", "event_type", "timestamp").count().filter("count > 1")

print("--- DATA QUALITY REPORT ---")
print(f"Total raw events: {total_raw_events}")
print(f"Duplicate scan events found: {duplicates.count()}")

sdf_dedup = sdf.dropDuplicates(["hub_id", "package_id", "event_type", "timestamp"])

pivoted_df = sdf_dedup.groupBy("hub_id", "package_id") \
    .pivot("event_type", ["ARRIVAL", "DEPARTURE"]) \
    .agg(F.min("timestamp"))

pivoted_df.show(5)

--- DATA QUALITY REPORT ---
Total raw events: 19482
Duplicate scan events found: 306
+-------+----------+-------------------+-------------------+
| hub_id|package_id|            ARRIVAL|          DEPARTURE|
+-------+----------+-------------------+-------------------+
|HUB_SUB|  PKG00306|2025-10-10 10:29:00|2025-10-10 13:52:54|
|HUB_SUB|  PKG00536|2025-10-08 12:04:00|2025-10-09 02:11:38|
|HUB_JKT|  PKG00558|2025-10-07 19:52:00|2025-10-07 20:18:52|
|HUB_SUB|  PKG01070|2025-10-10 10:48:00|2025-10-10 12:01:19|
|HUB_JKT|  PKG01428|2025-10-08 04:13:00|2025-10-08 08:15:18|
+-------+----------+-------------------+-------------------+
only showing top 5 rows


### Validasi Integritas Pasangan Scan

Tahap validasi menyaring data agar hanya paket dengan riwayat lengkap dan urutan waktu logis yang dihitung:
* **Missing ARRIVAL**: Paket hanya memiliki catatan keberangkatan tanpa kedatangan.
* **Missing DEPARTURE**: Paket hanya memiliki catatan kedatangan tanpa keberangkatan.
* **Invalid Sequence**: Waktu `DEPARTURE` tercatat lebih awal dari `ARRIVAL`.
* **Valid Pairs**: Pasangan scan yang valid (`DEPARTURE >= ARRIVAL`) untuk analisis dwell time.

In [4]:
total_pairs = pivoted_df.count()
missing_arrival = pivoted_df.filter(F.col("ARRIVAL").isNull() & F.col("DEPARTURE").isNotNull()).count()
missing_departure = pivoted_df.filter(F.col("ARRIVAL").isNotNull() & F.col("DEPARTURE").isNull()).count()
invalid_sequence = pivoted_df.filter(
    F.col("ARRIVAL").isNotNull() & F.col("DEPARTURE").isNotNull() & (F.col("DEPARTURE") < F.col("ARRIVAL"))
).count()

valid_pairs_df = pivoted_df.filter(
    F.col("ARRIVAL").isNotNull() & F.col("DEPARTURE").isNotNull() & (F.col("DEPARTURE") >= F.col("ARRIVAL"))
)

print(f"Total unique package-hub combinations: {total_pairs}")
print(f"- Missing ARRIVAL: {missing_arrival}")
print(f"- Missing DEPARTURE: {missing_departure}")
print(f"- Invalid Timestamp Sequence (DEPARTURE < ARRIVAL): {invalid_sequence}")
print(f"- Valid Pairs for Analysis: {valid_pairs_df.count()}")

Total unique package-hub combinations: 10000
- Missing ARRIVAL: 366
- Missing DEPARTURE: 458
- Invalid Timestamp Sequence (DEPARTURE < ARRIVAL): 291
- Valid Pairs for Analysis: 8885


## 4. Perhitungan Dwell Time dan Agregasi Metrik Hub

*Dwell Time* dihitung dalam satuan jam dengan selisih waktu antara `DEPARTURE` dan `ARRIVAL`:
$$\text{Dwell Time (jam)} = \frac{\text{Timestamp DEPARTURE} - \text{Timestamp ARRIVAL}}{3600}$$

Metrik yang dihitung per hub meliputi:
* `package_count`: Jumlah paket valid yang diproses.
* `avg_dwell_hours`: Rata-rata durasi transit (dalam jam).
* `median_dwell_hours`: Median durasi transit (dihitung menggunakan aproksimasi kuantil `percentile_approx`).

Hasil diurutkan secara menurun berdasarkan `avg_dwell_hours` untuk mengidentifikasi hub bottleneck.

In [5]:
processed_df = valid_pairs_df.withColumn(
    "dwell_hours",
    (F.col("DEPARTURE").cast("long") - F.col("ARRIVAL").cast("long")) / 3600.0
)

hub_metrics = processed_df.groupBy("hub_id").agg(
    F.count("package_id").alias("package_count"),
    F.round(F.mean("dwell_hours"), 2).alias("avg_dwell_hours"),
    F.round(F.percentile_approx("dwell_hours", 0.5), 2).alias("median_dwell_hours")
).orderBy(F.desc("avg_dwell_hours"))

hub_metrics.show(truncate=False)

+-------+-------------+---------------+------------------+
|hub_id |package_count|avg_dwell_hours|median_dwell_hours|
+-------+-------------+---------------+------------------+
|HUB_MKS|1492         |17.89          |12.68             |
|HUB_MES|1569         |11.48          |7.87              |
|HUB_BDG|1482         |4.01           |2.77              |
|HUB_JKT|1440         |3.96           |2.68              |
|HUB_SRG|1476         |3.91           |2.69              |
|HUB_SUB|1426         |3.83           |2.75              |
+-------+-------------+---------------+------------------+



## 5. Penyimpanan Output dan Pembuatan Dokumentasi Analisis

Hasil agregasi performa hub diekspor ke:
1. `output/top_bottleneck_hubs.csv`: File tabular ringkas berisi metrik utama per hub.
2. `docs/bigdata-bottleneck-analysis.md`: Laporan analisis terstruktur mencakup skema data, temuan kualitas, metrik dwell time, dan rekomendasi investigasi operasional.

In [6]:
os.makedirs("analysis", exist_ok=True)
os.makedirs("output", exist_ok=True)
os.makedirs("docs", exist_ok=True)

pandas_output = hub_metrics.toPandas()
pandas_output.to_csv("output/top_bottleneck_hubs.csv", index=False)

print("File top_bottleneck_hubs.csv berhasil disimpan di folder output/")

File top_bottleneck_hubs.csv berhasil disimpan di folder output/


In [ ]:
doc_content = f"""# Big Data Bottleneck Analysis - Scan Events

## 1. Dataset & Schema
Dataset `scan_events` mensimulasikan data logistik pelacakan paket (package tracking) dengan struktur kolom:
- `hub_id` (String): ID lokasi hub tempat scan dilakukan.
- `package_id` (String): ID unik dari paket yang dikirim.
- `event_type` (String): Jenis scan event, yaitu `ARRIVAL` (paket masuk hub) atau `DEPARTURE` (paket keluar hub).
- `timestamp` (Timestamp): Waktu terjadinya event scan.

## 2. Data Quality Check
Melalui analisis kualitas data yang komprehensif, ditemukan beberapa isu anomali dalam perekaman scan:
- **Duplicate Scan**: Ditemukan event scan ganda pada kombinasi kunci yang sama, dieliminasi menggunakan penanganan `dropDuplicates`.
- **Missing ARRIVAL**: Terdapat paket yang hanya merekam event `DEPARTURE` saja.
- **Missing DEPARTURE**: Paket yang masuk hub tetapi tidak pernah tercatat keluar.
- **Invalid Sequence (DEPARTURE < ARRIVAL)**: Paket yang tercatat keluar sebelum waktu kedatangannya.

Hanya data dengan pasangan ARRIVAL -> DEPARTURE yang lengkap dan berurutan secara logis yang digunakan untuk analisis lanjutan.

## 3. Hasil Perhitungan Dwell Time & Hub Aggregation
Hasil agregasi performa hub diurutkan berdasarkan `avg_dwell_hours` tertinggi:

"""

doc_content += pandas_output.to_markdown(index=False) + "\n\n"

top_hub = pandas_output.iloc[0]
second_hub = pandas_output.iloc[1]

doc_content += f"""## 4. Business Insight & Rekomendasi Investigasi

Based on the analysis, **{top_hub['hub_id']}** menunjukkan bottleneck terbesar dengan average dwell time mencapai **{top_hub['avg_dwell_hours']} jam** dan median sebesar **{top_hub['median_dwell_hours']} jam**, memproses total **{top_hub['package_count']} paket**. Angka ini jauh melebihi rata-rata standar operational level agreement (SLA) hub normal yang umumnya berada di bawah 5 jam.

Hal menarik lainnya adalah **{second_hub['hub_id']}** menempati posisi kedua dengan average dwell time **{second_hub['avg_dwell_hours']} jam**. Meskipun memiliki dwell time yang tinggi, tim operasional harus memprioritaskan investigasi pada **{top_hub['hub_id']}** terlebih dahulu karena tingginya akumulasi volume paket yang tertahan di sana.

### Rekomendasi Langkah Investigasi bagi Tim Operasional:
1. **Audit Alur Kerja Sortir**: Memeriksa kapasitas mesin penyortir (*sorting machine*) dan tata letak penempatan barang di **{top_hub['hub_id']}**.
2. **Evaluasi Penjadwalan Kurir/Armada**: Memastikan ketersediaan armada transportasi saat puncak kedatangan paket (peak arrival times).
3. **Analisis Shift Kerja**: Melakukan pengecekan produktivitas staf operasional selama jam-jam padat untuk menghindari penumpukan antrean.
"""

with open("docs/bigdata-bottleneck-analysis.md", "w") as f:
    f.write(doc_content)

print("Dokumen docs/bigdata-bottleneck-analysis.md berhasil dibuat!")

Dokumen docs/bigdata-bottleneck-analysis.md berhasil dibuat!
